# Lumbar Spine Detection — Vertebrae Pointing (L1–L5)

This notebook detects the lumbar spine in sagittal T2 MRI and **points to / labels every
vertebra correctly (L1, L2, L3, L4, L5)** using the trained SPIDER model from
`spine-foundation`.

The model returns 10 landmarks: the **5 vertebra centres first (L1..L5)**, then the
**5 disc centres (L1/L2..L5/S1)**. This notebook draws both, labelled, on the original
image.

**Data:** the SPIDER dataset lives in your Google Drive. If a trained `best_model.pth`
already exists in `checkpoints/` it is used directly; otherwise we train a *localizer*
first so the model can point at vertebrae.

In [ ]:
GITHUB_URL = 'https://github.com/codermisba/spinelit-ai.git'  # <-- your repo
print('Repo:', GITHUB_URL)

In [ ]:
# 1. GPU check (Runtime > Change runtime type > GPU)
!nvidia-smi -L || echo 'Please switch to a GPU runtime'

In [ ]:
# 2. Mount Google Drive (where the SPIDER dataset lives)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone / refresh the repo
%cd /content
import os
GITHUB_URL = GITHUB_URL or 'https://github.com/codermisba/spinelit-ai.git'
if not os.path.exists('/content/spine-foundation'):
    !git clone {GITHUB_URL} spine-foundation
%cd /content/spine-foundation/
!git checkout -- . 2>/dev/null; git pull || echo '(pull skipped)'
print('repo ready at /content/spine-foundation')

In [ ]:
# 4. Install dependencies
!pip install -q -r requirements.txt
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 5. Prepare the SPIDER data from Drive

Point `DATA_DIR` at the raw SPIDER folder in Drive that contains the
`images/` + `masks/` subfolders (plus `radiological_gradings.csv`), then build the
mid-sagittal JPGs + landmark CSV. Set `SKIP_PREP = True` if a prepared dataset already
exists in the repo (you ran this notebook before).

In [ ]:
import os, shutil

SPIDER_DIR = '/content/drive/MyDrive/SPIDER_data'  # <-- EDIT: raw SPIDER folder in Drive
SKIP_PREP   = False                                 # set True if JPGs + coords already exist

if not SKIP_PREP:
    # Remove any stale prepared data so prep rewrites it from your Drive copy
    shutil.rmtree('dataset/data/processed_spider_jpgs', ignore_errors=True)
    for f in ['coords_pretrain.csv', 'ddd_labels.csv']:
        p = 'dataset/' + f
        if os.path.exists(p):
            os.remove(p)

    # If the raw images/ + masks/ are ALREADY extracted in Drive, skip the
    # Zenodo download. Otherwise prepare_spider downloads + extracts for you.
    has_raw = os.path.isdir(os.path.join(SPIDER_DIR, 'images'))
    flag = '--skip_download' if has_raw else ''
    !python prepare_spider.py --data_dir "{SPIDER_DIR}" {flag}
else:
    print('SKIP_PREP=True - reusing already-prepared data')

print('jpg count:', len(os.listdir('dataset/data/processed_spider_jpgs')))

## 6. (Optional) Train a localizer if no checkpoint exists

The vertebra-coordinate head is trained automatically by `train.py`. If a
`checkpoints/best_model.pth` already exists, skip to **Section 7**.

In [ ]:
import os
if not os.path.exists('checkpoints/best_model.pth'):
    print('No checkpoint found - training a localizer (60 epochs)...')
    import re
    cfg = open('config.py').read()
    cfg = cfg.replace('IMAGE_SIZE = 256', 'IMAGE_SIZE = 512')
    cfg = cfg.replace('BATCH_SIZE = 4', 'BATCH_SIZE = 32')
    cfg = cfg.replace('NUM_WORKERS = 0', 'NUM_WORKERS = 2')
    open('config.py','w').write(cfg)
    !python train.py --epochs 60
else:
    print('Checkpoint exists: checkpoints/best_model.pth')
    print('Skipping training.')

## 7. Run vertebra detection on SPIDER images

Loads the trained model and, for every (or a chosen) mid-sagittal scan, detects and
labels the 5 lumbar vertebra centres. Choose which images with `N_MAX` (set to a small
number like 3 to spot-check, or `None` for all).

In [ ]:
import glob, os, torch
from PIL import Image
import numpy as np
from config import IMAGE_SIZE, VERTEBRAE, DISC_LEVELS
from model import SpineFoundationModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_MAX  = 3            # set None to see ALL scans
SHOW_VERTS = True     # draw + label the 5 vertebra centres (L1..L5)
SHOW_DISCS = True     # also draw the 5 disc centres (context)

ckpt = 'checkpoints/best_model.pth'
assert os.path.exists(ckpt), 'No checkpoint - run Section 6 first'

model = SpineFoundationModel().to(DEVICE).eval()
state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
model.load_state_dict(state['model_state_dict'])
print('Loaded model:', ckpt)

jpgs = sorted(glob.glob('dataset/data/processed_spider_jpgs/*.jpg'))
if N_MAX:
    jpgs = jpgs[:N_MAX]
print('images to process:', len(jpgs))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# Colour per vertebra, colourblind-friendly
VERT_COL = {'L1':'#d62728','L2':'#ff7f0e','L3':'#2ca02c',
            'L4':'#1f77b4','L5':'#9467bd'}
DISC_COL = '#555555'

@torch.no_grad()
def predict_landmarks(path):
    img   = Image.open(path).convert('RGB')
    w, h  = img.size                       # original pixel size
    t     = torch.from_numpy(np.array(img.resize((IMAGE_SIZE, IMAGE_SIZE),
                                                 Image.BILINEAR)) / 255.0)
    x     = t.permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
    out   = model(x)
    coords = out['coords'][0].view(10, 2).cpu().numpy()   # normalized 0-1
    conf   = out['localization_conf'][0].cpu().numpy()
    # Map normalized (image coords, y down) -> original pixel space
    px = coords.copy()
    px[:, 0] *= w
    px[:, 1] *= h
    return img, px, conf

def draw(path):
    img, px, conf = predict_landmarks(path)
    fig, ax = plt.subplots(figsize=(6.5, 9))
    ax.imshow(img, cmap='gray')
    n_v = len(VERTEBRAE)
    if SHOW_VERTS:
        for i, v in enumerate(VERTEBRAE):
            x, y = px[i]
            ax.plot(x, y, 'o', ms=13, mfc=VERT_COL[v], mec='white', mew=1.5,
                    zorder=5)
            ax.text(x+8, y-6, f'{v}\nconf {conf[i]:.2f}', color=VERT_COL[v],
                    fontweight='bold', fontsize=10, zorder=6)
    if SHOW_DISCS:
        for i, d in enumerate(DISC_LEVELS):
            x, y = px[n_v + i]
            ax.plot(x, y, 's', ms=9, mfc=DISC_COL, mec='white', mew=1.0,
                    zorder=4)
            ax.text(x+8, y+6, d, color=DISC_COL, fontsize=9, zorder=6)
    ax.set_title(f'{os.path.basename(path)}  (image {img.size[0]}x{img.size[1]}px)')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    return img, px, conf

results = {}
for p in jpgs:
    img, px, conf = draw(p)
    results[os.path.basename(p)] = (img, px, conf)


## 8. Export the detected points to a CSV

Save every vertebra / disc landmark (original pixel coordinates + confidence) for the
images you processed, so you can reuse them downstream (`detected_vertebrae.csv`).

In [ ]:
import pandas as pd

rows = []
for fname, (img, px, conf) in results.items():
    n_v = len(VERTEBRAE)
    for i, v in enumerate(VERTEBRAE):
        rows.append({'filename': fname, 'kind': 'vertebra', 'level': v,
                     'x': float(px[i][0]), 'y': float(px[i][1]),
                     'conf': float(conf[i])})
    for i, d in enumerate(DISC_LEVELS):
        rows.append({'filename': fname, 'kind': 'disc', 'level': d,
                     'x': float(px[n_v+i][0]), 'y': float(px[n_v+i][1]),
                     'conf': float(conf[n_v+i])})

df = pd.DataFrame(rows)
df.to_csv('detected_vertebrae.csv', index=False)
print('Saved', len(df), 'landmarks to detected_vertebrae.csv')
df.head(10)

## 9. (Optional) Verify localization quality on the validation split

Report mean vertebra / disc localization error (px) so you know how accurately the
model points at each vertebra.

In [ ]:
!python evaluate.py